<a href="https://colab.research.google.com/github/maliaheupel/Capstone-ACC/blob/main/notebooks/2026_11_20_%E2%80%94_Capstone_Proposal_and_Pipeline_Setup_%E2%80%94_Check_In.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Capstone Proposal and Pipeline Setup

**Check-In — 2026-11-20 · Fall 2026**  

---

## Proposal and pipeline setup: week 1 check-in

This notebook is not graded and nothing in it is submitted. The only graded work in the capstone is the final submission on Friday, December 18. Your team should still work through every section by Sunday, November 22, because each one is a piece of the final notebook, and a team that reaches December with no data loaded will not be able to catch up.

Keep this notebook in your team repository and commit it as you go. A modest pipeline that loads real rows into SQLite is worth more at this point than an ambitious plan with nothing behind it. If a step is not working yet, leave the actual error in the notebook along with a note about what you tried, so the team can pick it up together.

There is no class on Monday, November 23, so treat this weekend as the time to get the data flowing. The week after Thanksgiving is for analysis.

### 1. Proposal

**a)** Which ACC team or teams does your group expect to be the most interesting story, and why? Base the answer on something you saw in the data on Wednesday.

**b)** Write a hypothesis you can test, such as a claim that one team's record is better than its rating or that a particular team has faced the weakest schedule.

**c)** What result in the data would show that your hypothesis is wrong?

a) Virginia and Duke will be the most interesting story. Virginia is currently undefeated and ranked 6th nationally. Even though Duke has 2 more losses than Virginia, they are ranked 5th nationally. It will be interesting to compare an undefeated record with an opponent-adjusted ranking.
b) Also a team may be undefeated early in the season, they usually build that score against weaker teams, so their average scoring margin will drop as they face more competitive teams.
c) Our hypothesis will be wrong if undefeated teams maintain their victory margin even as the strength of their competitors increase.

### 2. Raw pulls saved

**TODO:** show that every raw response is on disk. Print how many Torvik snapshots, how many scoreboard dates, and how many game summaries you have saved, along with the date range the scoreboard files cover.

In [1]:
# TODO
import json
import os
import pandas as pd
import requests

# creates scoreboards folder
os.makedirs("scoreboards", exist_ok=True)
os.makedirs("summaries", exist_ok=True)

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
summary_url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary"
dates = pd.date_range("2026-11-01", "2026-11-22").strftime("%Y%m%d")

for d in dates:
  r = requests.get(
      url, params={"groups": 2, "dates": d, "limit": 200}, timeout=15
  )
  with open(f"scoreboards/{d}.json", "w") as f:
    f.write(r.text)

# removes any empty files so they get redownloaded
for folder in ["scoreboards", "summaries"]:
  for fn in os.listdir(folder):
    if os.path.getsize(f"{folder}/{fn}") == 0:
      os.remove(f"{folder}/{fn}")

  for filename in os.listdir("scoreboards"):
    with open(f"scoreboards/{filename}") as f:
      data = json.load(f)
    for event in data["events"]:
      gid = event["id"]
      if not os.path.exists(f"summaries/{gid}.json"):
        sr = requests.get(summary_url, params={"event": gid}, timeout=15)
        with open(f"summaries/{gid}.json", "w") as sf:
          sf.write(sr.text)

# counts torvik's csv snapshots into root directory
torvik_count = 0
for f in os.listdir("."):
  if f.endswith(".csv"):
    torvik_count += 1

print("Torvik snapshots saved:", torvik_count)
print("Scoreboard dates saved:", len(os.listdir("scoreboards")))
print("Game summaries saved:", len(os.listdir("summaries")))
print("Date range covered:", dates[0], "to", dates[-1])

Torvik snapshots saved: 1
Scoreboard dates saved: 22
Game summaries saved: 107
Date range covered: 20261101 to 20261122


### 3. The name crosswalk

**TODO:** match ESPN's team names to Torvik's. Print the names that did not match after your first automatic attempt, then the names that still do not match after your fixes. Finish with an assertion that all 18 ACC schools are matched, and a second assertion that no two ESPN teams point to the same Torvik row.

Some opponents are not in Division I and have no rating. Decide how your project will treat them, and record that decision in the log below.

In [2]:
# TODO
import json

df_torvik = pd.read_csv("torvik_2026-11-23.csv")
torvik_teams = set(df_torvik["team"])
acc_teams = set(df_torvik[df_torvik["conf"] == "ACC"]["team"])

# collects unique teams from ESPN
espn_teams = {}
for filename in os.listdir("scoreboards"):
  with open(f"scoreboards/{filename}", "r") as f:
    data = json.load(f)
  for event in data["events"]:
    for comp in event["competitions"][0]["competitors"]:
      t = comp["team"]
      espn_teams[t["id"]] = t["location"]

# print names that did not match on the first attempt
unmatched_1 = sorted(
    [name for name in set(espn_teams.values()) if name not in torvik_teams]
)
print("Unmatched after first attempt:", unmatched_1)

# fixes name differences
fixes = {
    "NC State": "N.C. State",
    "Miami": "Miami FL",
    "UConn": "Connecticut",
    "Ole Miss": "Mississippi",
    "Cal": "California",
    "Queens University": "Queens",
    "App State": "Appalachian St.",
    "Saint Francis": "St. Francis PA",
    "Sam Houston": "Sam Houston St.",
    "Miami (OH)": "Miami OH",
    "Loyola Maryland": "Loyola MD",
    "Gardner-Webb": "Gardner Webb",
    "California Baptist": "Cal Baptist",
    "Florida International": "FIU",
    "American University": "American",
    "Texas A&M-Corpus Christi": "Texas A&M Corpus Chris",
}

crosswalk = {}
unmatched_2 = []

for espn_id, name in espn_teams.items():
  if name in torvik_teams:
    crosswalk[espn_id] = name
  elif name in fixes:
    crosswalk[espn_id] = fixes[name]
  elif name.endswith(" State") and (name[:-6] + " St.") in torvik_teams:
    crosswalk[espn_id] = name[:-6] + " St."
  else:
    crosswalk[espn_id] = None
    unmatched_2.append(name)

print("Still unmatched after fixes (non-D1):", sorted(set(unmatched_2)))

# checks
matched = [v for v in crosswalk.values() if v is not None]
assert acc_teams.issubset(set(matched)), "Missing an ACC team"
assert len(matched) == len(set(matched)), "Duplicate Torvik match"
print("PASS: All 18 ACC schools matched uniquely!")

Unmatched after first attempt: ['Adams State', 'American University', 'Ball State', 'Coppin State', 'East Tennessee State', 'Florida State', 'Loyola Maryland', 'Miami', 'Michigan State', 'Morehead State', 'Murray State', 'NC State', 'Norfolk State', 'Oklahoma State', 'Penn State', 'Sacramento State', 'South Carolina State', 'Texas A&M-Corpus Christi', 'Washington State']
Still unmatched after fixes (non-D1): ['Adams State']
PASS: All 18 ACC schools matched uniquely!


### 4. Tables in SQLite

**TODO:** create your tables with primary keys and load them. Print the schema stored in `sqlite_master` and the row count of each table. At this stage you should have at least `teams`, `team_crosswalk`, `ratings`, and `games`.

In [3]:
# TODO
import sqlite3

def load_db():
  conn = sqlite3.connect("acc.db")
  cur = conn.cursor()

  # creates tables with primary keys
  cur.executescript("""
    CREATE TABLE IF NOT EXISTS teams (
        torvik_team TEXT PRIMARY KEY,
        conf TEXT
    );
    CREATE TABLE IF NOT EXISTS team_crosswalk (
        espn_id TEXT PRIMARY KEY,
        espn_name TEXT,
        torvik_team TEXT
    );
    CREATE TABLE IF NOT EXISTS ratings (
        snapshot_date TEXT,
        torvik_team TEXT,
        record TEXT,
        barthag REAL,
        barthag_rank INTEGER,
        adjoe REAL,
        adjde REAL,
        PRIMARY KEY (snapshot_date, torvik_team)
    );
    CREATE TABLE IF NOT EXISTS games (
        game_id TEXT PRIMARY KEY,
        game_date_et TEXT,
        home_team TEXT,
        away_team TEXT,
        home_score INTEGER,
        away_score INTEGER
    );
    CREATE TABLE IF NOT EXISTS team_games (
        game_id TEXT,
        team_name TEXT,
        home_away TEXT,
        points_for INTEGER,
        points_against INTEGER,
        margin INTEGER,
        fg TEXT,
        three_pt TEXT,
        ft TEXT,
        rebounds INTEGER,
        assists INTEGER,
        steals INTEGER,
        blocks INTEGER,
        turnovers INTEGER,
        estimated_possessions REAL,
        stats_json TEXT,
        PRIMARY KEY (game_id, team_name)
    );
    CREATE TABLE IF NOT EXISTS player_games (
        game_id TEXT,
        player_id TEXT,
        player_name TEXT,
        team_name TEXT,
        minutes INTEGER,
        points INTEGER,
        rebounds INTEGER,
        assists INTEGER,
        steals INTEGER,
        blocks INTEGER,
        turnovers INTEGER,
        fg TEXT,
        three_pt TEXT,
        ft TEXT,
        stats_json TEXT,
        PRIMARY KEY (game_id, player_id)
    );
    """)

  # loads teams and ratings from Torvik
  for f in os.listdir("."):
    if f.endswith(".csv"):
      snap_date = f.replace("torvik_", "").replace(".csv", "")
      df = pd.read_csv(f)
      for _, row in df.iterrows():
        cur.execute(
            "INSERT OR REPLACE INTO teams VALUES (?, ?)",
            (row["team"], row["conf"]),
        )
        cur.execute(
            "INSERT OR REPLACE INTO ratings VALUES (?, ?, ?, ?, ?, ?, ?)",
            (
                snap_date,
                row["team"],
                row["record"],
                row["barthag"],
                row["rank.1"],
                row["adjoe"],
                row["adjde"],
            ),
        )

  # loads team_crosswalk
  for espn_id, espn_name in espn_teams.items():
    cur.execute(
        "INSERT OR REPLACE INTO team_crosswalk VALUES (?, ?, ?)",
        (espn_id, espn_name, crosswalk[espn_id]),
    )

  # loads finished games from scoreboards (converting UTC to EST)
  for filename in os.listdir("scoreboards"):
    with open(f"scoreboards/{filename}") as f:
      data = json.load(f)
    for event in data["events"]:
      comp = event["competitions"][0]
      if not comp["status"]["type"]["completed"]:
        continue

      date_et = (
          pd.to_datetime(comp["date"])
          .tz_convert("America/New_York")
          .strftime("%Y-%m-%d")
      )

      t0, t1 = comp["competitors"]
      home = t0 if t0["homeAway"] == "home" else t1
      away = t1 if t0["homeAway"] == "home" else t0

      home_name = crosswalk.get(home["team"]["id"]) or home["team"]["location"]
      away_name = crosswalk.get(away["team"]["id"]) or away["team"]["location"]

      cur.execute(
          "INSERT OR REPLACE INTO games VALUES (?, ?, ?, ?, ?, ?)",
          (
              event["id"],
              date_et,
              home_name,
              away_name,
              int(home["score"]),
              int(away["score"]),
          ),
      )

  # loads team_games and player_games with unpacked columns from summaries
  for filename in os.listdir("summaries"):
    gid = filename.replace(".json", "")
    with open(f"summaries/{filename}") as f:
      sdata = json.load(f)

    box = sdata.get("boxscore", {})

    g_row = cur.execute(
        "SELECT home_team, away_team, home_score, away_score FROM games WHERE"
        " game_id = ?",
        (gid,),
    ).fetchone()

    for team_entry in box.get("teams", []):
      t_info = team_entry["team"]
      t_name = crosswalk.get(t_info["id"]) or t_info["location"]
      home_away = team_entry.get("homeAway", "")
      s = {
          stat["name"]: stat["displayValue"]
          for stat in team_entry.get("statistics", [])
      }

      if g_row:
        pts_for = g_row[2] if g_row[0] == t_name else g_row[3]
        pts_against = g_row[3] if g_row[0] == t_name else g_row[2]
        margin = pts_for - pts_against
      else:
        pts_for, pts_against, margin = None, None, None

      fg_str = s.get("fieldGoalsMade-fieldGoalsAttempted", "0-0")
      ft_str = s.get("freeThrowsMade-freeThrowsAttempted", "0-0")
      fga = int(fg_str.split("-")[1]) if "-" in fg_str else 0
      fta = int(ft_str.split("-")[1]) if "-" in ft_str else 0
      oreb = int(s.get("offensiveRebounds", 0) or 0)
      to = int(s.get("turnovers", 0) or 0)
      est_poss = round(fga - oreb + to + 0.475 * fta, 1)

      cur.execute(
          "INSERT OR REPLACE INTO team_games VALUES"
          " (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
          (
              gid,
              t_name,
              home_away,
              pts_for,
              pts_against,
              margin,
              fg_str,
              s.get("threePointFieldGoalsMade-threePointFieldGoalsAttempted"),
              ft_str,
              int(s.get("totalRebounds", 0) or 0),
              int(s.get("assists", 0) or 0),
              int(s.get("steals", 0) or 0),
              int(s.get("blocks", 0) or 0),
              to,
              est_poss,
              json.dumps(s),
          ),
      )

    for p_team in box.get("players", []):
      t_info = p_team["team"]
      t_name = crosswalk.get(t_info["id"]) or t_info["location"]
      if not p_team.get("statistics"):
        continue
      labels = p_team["statistics"][0].get("names", [])
      for athlete_row in p_team["statistics"][0].get("athletes", []):
        ath = athlete_row["athlete"]
        ps = dict(zip(labels, athlete_row.get("stats", [])))

        def to_int(val):
          try:
            return int(val)
          except (ValueError, TypeError):
            return 0

        cur.execute(
            "INSERT OR REPLACE INTO player_games VALUES"
            " (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (
                gid,
                ath["id"],
                ath["displayName"],
                t_name,
                to_int(ps.get("MIN")),
                to_int(ps.get("PTS")),
                to_int(ps.get("REB")),
                to_int(ps.get("AST")),
                to_int(ps.get("STL")),
                to_int(ps.get("BLK")),
                to_int(ps.get("TO")),
                ps.get("FG"),
                ps.get("3PT"),
                ps.get("FT"),
                json.dumps(ps),
            ),
        )

  conn.commit()

  # prints
  for name, sql in cur.execute(
      "SELECT name, sql FROM sqlite_master WHERE type='table'"
  ):
    print(f"\n--- {name} ---\n{sql}")

  print("\n--- Row Counts ---")
  for tbl in [
      "teams",
      "team_crosswalk",
      "ratings",
      "games",
      "team_games",
      "player_games",
  ]:
    count = cur.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"{tbl}: {count}")

  conn.close()


load_db()


--- teams ---
CREATE TABLE teams (
        torvik_team TEXT PRIMARY KEY,
        conf TEXT
    )

--- team_crosswalk ---
CREATE TABLE team_crosswalk (
        espn_id TEXT PRIMARY KEY,
        espn_name TEXT,
        torvik_team TEXT
    )

--- ratings ---
CREATE TABLE ratings (
        snapshot_date TEXT,
        torvik_team TEXT,
        record TEXT,
        barthag REAL,
        barthag_rank INTEGER,
        adjoe REAL,             
        adjde REAL,
        PRIMARY KEY (snapshot_date, torvik_team)
    )

--- games ---
CREATE TABLE games (
        game_id TEXT PRIMARY KEY,
        game_date_et TEXT,
        home_team TEXT,
        away_team TEXT,
        home_score INTEGER,
        away_score INTEGER
    )

--- team_games ---
CREATE TABLE team_games (
        game_id TEXT,
        team_name TEXT,
        home_away TEXT,
        points_for INTEGER,
        points_against INTEGER,
        margin INTEGER,
        fg TEXT,
        three_pt TEXT,
        ft TEXT,
        rebounds INTE

### 5. The rerun test

**TODO:** run your load a second time and show that every row count is unchanged. If the counts grow, the load is appending duplicates, and every average you compute later will be wrong in a way that is hard to notice.

In [4]:
# TODO
load_db()


--- teams ---
CREATE TABLE teams (
        torvik_team TEXT PRIMARY KEY,
        conf TEXT
    )

--- team_crosswalk ---
CREATE TABLE team_crosswalk (
        espn_id TEXT PRIMARY KEY,
        espn_name TEXT,
        torvik_team TEXT
    )

--- ratings ---
CREATE TABLE ratings (
        snapshot_date TEXT,
        torvik_team TEXT,
        record TEXT,
        barthag REAL,
        barthag_rank INTEGER,
        adjoe REAL,             
        adjde REAL,
        PRIMARY KEY (snapshot_date, torvik_team)
    )

--- games ---
CREATE TABLE games (
        game_id TEXT PRIMARY KEY,
        game_date_et TEXT,
        home_team TEXT,
        away_team TEXT,
        home_score INTEGER,
        away_score INTEGER
    )

--- team_games ---
CREATE TABLE team_games (
        game_id TEXT,
        team_name TEXT,
        home_away TEXT,
        points_for INTEGER,
        points_against INTEGER,
        margin INTEGER,
        fg TEXT,
        three_pt TEXT,
        ft TEXT,
        rebounds INTE

### 6. Decision log

**TODO:** list every decision so far that changed the data, with the number of rows each one affected. The crosswalk fixes, the time zone conversion, and the handling of unfinished or unrated games all belong here.

In [5]:
# TODO
conn = sqlite3.connect("acc.db")
cur = conn.cursor()

# counts crosswalk name fixes (where ESPN name != Torvik name)
name_fixes = cur.execute(
    "SELECT COUNT(*) FROM team_crosswalk WHERE torvik_team IS NOT NULL AND"
    " espn_name != torvik_team"
).fetchone()[0]

# 2. counts unrated / non-D1 teams and games involving them
unrated_teams = cur.execute(
    "SELECT COUNT(*) FROM team_crosswalk WHERE torvik_team IS NULL"
).fetchone()[0]
unrated_games = cur.execute("""
    SELECT COUNT(*) FROM games
    WHERE home_team IN (SELECT espn_name FROM team_crosswalk WHERE torvik_team IS NULL)
       OR away_team IN (SELECT espn_name FROM team_crosswalk WHERE torvik_team IS NULL)
""").fetchone()[0]

# 3. counts games whose UTC date differed from their Eastern Time date and skips
tz_shifted = 0
unfinished = 0
for filename in os.listdir("scoreboards"):
  with open(f"scoreboards/{filename}") as f:
    data = json.load(f)
  for event in data["events"]:
    comp = event["competitions"][0]
    if not comp["status"]["type"]["completed"]:
      unfinished += 1
      continue
    utc_date = comp["date"][:10]
    et_date = (
        pd.to_datetime(comp["date"])
        .tz_convert("America/New_York")
        .strftime("%Y-%m-%d")
    )
    if utc_date != et_date:
      tz_shifted += 1

# counts total rows loaded into ratings and games
ratings_rows = cur.execute("SELECT COUNT(*) FROM ratings").fetchone()[0]
games_rows = cur.execute("SELECT COUNT(*) FROM games").fetchone()[0]
conn.close()

# builds and displays the decision log
decisions = [
    {
        "Decision": (
            "Crosswalk name fixes (e.g., NC State -> N.C. State, Miami -> Miami"
            " FL)"
        ),
        "Rows Affected": name_fixes,
    },
    {
        "Decision": (
            f"Keep unrated non-D1 opponents ({unrated_teams} team) with"
            " torvik_team = NULL"
        ),
        "Rows Affected": unrated_games,
    },
    {
        "Decision": (
            "Convert ESPN game timestamps from UTC to Eastern Time"
            " (America/New_York)"
        ),
        "Rows Affected": tz_shifted,
    },
    {
        "Decision": "Filter out unfinished/future games (completed == False)",
        "Rows Affected": unfinished,
    },
    {
        "Decision": (
            "Use rank.1 as barthag_rank and drop duplicate rank column in Torvik"
        ),
        "Rows Affected": ratings_rows,
    },
    {
        "Decision": "Cast ESPN string scores to integers",
        "Rows Affected": games_rows,
    },
    {
        "Decision": (
            "Include Torvik offensive and defensive efficiency (adjoe, adjde)"
            " in ratings"
        ),
        "Rows Affected": ratings_rows,
    },
]

pd.DataFrame(decisions)

,Decision,Rows Affected
0,"Crosswalk name fixes (e.g., NC State -> N.C. S...",18
1,Keep unrated non-D1 opponents (1 team) with to...,1
2,Convert ESPN game timestamps from UTC to Easte...,77
3,Filter out unfinished/future games (completed ...,0
4,Use rank.1 as barthag_rank and drop duplicate ...,365
5,Cast ESPN string scores to integers,107
6,Include Torvik offensive and defensive efficie...,365


### 7. Self-check

In [6]:
setup = {
    'torvik_snapshots_saved': 1,      # TODO: an integer
    'scoreboard_dates_saved': 22,      # TODO: an integer
    'acc_teams_matched': 18,           # TODO: should be 18
    'games_table_rows': 107,            # TODO: an integer
    'rerun_counts_unchanged': True,     # TODO: True after the test in section 5
    'decisions_logged': 7,            # TODO: how many entries
    'repo_url': 'https://github.com/maliaheupel/Capstone-ACC',
    'notebook_runs_clean': True,        # TODO: True after Restart and Run All
}

for label, value in setup.items():
    ok = not (value is None or value is False
              or (isinstance(value, str) and value.startswith('TODO')))
    print('PASS' if ok else 'FAIL', '-', label, '=', value)

PASS - torvik_snapshots_saved = 1
PASS - scoreboard_dates_saved = 22
PASS - acc_teams_matched = 18
PASS - games_table_rows = 107
PASS - rerun_counts_unchanged = True
PASS - decisions_logged = 7
PASS - repo_url = https://github.com/maliaheupel/Capstone-ACC
PASS - notebook_runs_clean = True


### 8. Plan for the break

**a)** Question 6 compares ratings taken before each game with the result of the game, which only works if your team saves a new ratings snapshot regularly from now on. Who will take the snapshots during Thanksgiving week, and on which days?

**b)** What is the most likely thing to go wrong before November 30, and what is your team doing about it?

a) Malia will take snapshots during Thanksgiving break everyday (Mon-Fri) in the morning.
b) The most likely issue to arise is forgetting to take the snapshots during break due to the busy holiday season. We are setting daily reminders and checking in with the group to act as a reminder.